# Clase 4 — Dar herramientas a un agente

En la Clase 3 usamos un LLM local para clasificar tickets. El modelo interpretaba texto y devolvía una decisión.

Hoy agregaremos una capacidad nueva: el modelo podrá **pedir que Python ejecute una función**.

> El LLM elige la herramienta. Python conserva la función y decide si la ejecuta.

## Objetivos

- Construir una función breve y fácil de explicar.
- Describir esa función dentro del `system prompt`.
- Hacer que el LLM local decida cuándo utilizarla.
- Ejecutar únicamente herramientas registradas.

---
## Agenda

### Explicación guiada — 40 minutos

1. Construir `consultar_menu()` paso a paso.
2. Explicar la herramienta en el `system prompt`.
3. Observar cómo el LLM decide usarla.
4. Permitir que Python la ejecute.

### Ejercicio — 40 minutos

Crear dos herramientas para el agente de tickets de la Clase 3:

- consultar las colas disponibles;
- asignar un ticket a una cola.

---
# Parte 1 — Una herramienta sencilla

## El caso del restaurante

Una persona pregunta qué puede comer. Nuestro agente no tiene esa información dentro del modelo: debe consultarla en Python.

La herramienta será una función llamada `consultar_menu()`.

## Paso 1 — Definir los datos

El menú será un diccionario pequeño para que podamos concentrarnos en el mecanismo de herramientas.

In [ ]:
MENU = {
    "empanadas": 9000,
    "pizza": 12000,
    "ensalada": 7500,
}

## Paso 2 — Escribir la función

Una buena herramienta realiza una tarea concreta, tiene un nombre claro y devuelve un resultado predecible.

In [ ]:
def consultar_menu():
    return {"ok": True, "menu": MENU}

In [ ]:
consultar_menu()

### Leamos las dos líneas

- `def consultar_menu()` crea una función sin parámetros.
- `return` devuelve un diccionario con el resultado.

La función no conoce al LLM y el LLM todavía no conoce la función.

---
## Paso 3 — Registrar la herramienta

El registro define qué funciones tiene permitido ejecutar el programa.

In [ ]:
HERRAMIENTAS_RESTAURANTE = {
    "consultar_menu": consultar_menu,
}

Registrar una función es distinto de describírsela al modelo:

- Python necesita el objeto `consultar_menu`.
- El LLM necesita su nombre, propósito y parámetros.

No copiamos el código de la función dentro del prompt. Solo explicamos cómo pedir su uso.

---
## Paso 4 — Incluir la herramienta en el `system prompt`

El contrato de salida será el mismo en todos los casos:

```json
{"herramienta": "nombre o null", "argumentos": {}}
```

In [ ]:
SYSTEM_PROMPT_RESTAURANTE = """
Sos el asistente de un restaurante.

HERRAMIENTA DISPONIBLE:
- consultar_menu: muestra los platos y precios. No recibe parámetros.

Si necesitás la herramienta, respondé solo:
{"herramienta": "consultar_menu", "argumentos": {}}

Si no la necesitás, respondé solo:
{"herramienta": null, "argumentos": {}}
""".strip()

### Las tres preguntas que responde la descripción

1. ¿Cómo se llama la herramienta?
2. ¿Para qué sirve?
3. ¿Qué parámetros necesita?

Con eso alcanza para este ejemplo.

---
## Paso 5 — Usar el LLM local de la Clase 3

Reutilizamos exactamente `LFM2.5-1.2B-Instruct` con `llama_cpp`.

In [ ]:
import json
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

ruta_modelo = hf_hub_download(
    repo_id="unsloth/LFM2.5-1.2B-Instruct-GGUF",
    filename="LFM2.5-1.2B-Instruct-Q8_0.gguf",
)
llm = Llama(model_path=ruta_modelo, n_ctx=4096,
            n_gpu_layers=0, verbose=False)

La función siguiente solamente envía el `system prompt` y el mensaje. Si el modelo no genera JSON válido, devuelve un error visible en vez de detener el notebook.

In [ ]:
def decidir_con_llm(mensaje, system_prompt):
    salida = llm.create_chat_completion(
        messages=[{"role": "system", "content": system_prompt},
                  {"role": "user", "content": mensaje}],
        response_format={"type": "json_object"},
        temperature=0, max_tokens=80,
    )
    texto = salida["choices"][0]["message"]["content"].strip()
    try:
        return json.loads(texto)
    except json.JSONDecodeError:
        return {"herramienta": None, "argumentos": {},
                "error": "salida JSON inválida", "salida_cruda": texto}

In [ ]:
mensaje = "¿Qué tienen para comer?"
decision = decidir_con_llm(mensaje, SYSTEM_PROMPT_RESTAURANTE)
print("CLIENTE:", mensaje)
print("DECISIÓN:", decision, "\n")

Hasta aquí el modelo solo produjo un diccionario. La función todavía no se ejecutó.

---
## Paso 6 — Ejecutar la decisión

Python busca el nombre propuesto dentro del registro. Si no está permitido, no ejecuta nada.

In [ ]:
def ejecutar(decision, herramientas):
    nombre = decision.get("herramienta")
    argumentos = decision.get("argumentos", {})

    if nombre not in herramientas:
        return {"ok": False, "error": "herramienta no permitida"}

    return herramientas[nombre](**argumentos)

In [ ]:
mensaje = "¿Qué tienen para comer?"
decision = decidir_con_llm(mensaje, SYSTEM_PROMPT_RESTAURANTE)
resultado = ejecutar(decision, HERRAMIENTAS_RESTAURANTE)

print("DECISIÓN:", decision)
print("RESULTADO:", resultado)

### El recorrido completo

```text
pregunta → system prompt → LLM elige → Python verifica → función responde
```

El prompt comunica posibilidades. El registro de Python concede capacidades reales.

---
# Parte 2 — Ejercicio: herramientas para clasificar tickets

Retomamos el caso de la Clase 3. El agente debe poder consultar las colas de pedidos disponibles y asignar un ticket a una de ellas.

Construirán dos herramientas:

1. `consultar_colas()` — devuelve las colas y su propósito.
2. `asignar_ticket(ticket_id, cola)` — registra la asignación si el ticket y la cola existen.

Pueden usar Claude, ChatGPT o Gemini y tomar `consultar_menu()` como ejemplo, pero deberán comprender y explicar el código final.

## Datos iniciales

In [ ]:
from pathlib import Path

COLAS = {
    "seguridad": "compras o accesos no reconocidos",
    "pagos": "cobros, rechazos y reintegros",
    "logistica": "entregas demoradas, dañadas o incompletas",
    "general": "otras consultas",
}
TICKETS = {"T001": "Me cobraron dos veces",
           "T002": "Mi pedido todavía no llegó"}

ARCHIVO_ASIGNACIONES = Path("asignaciones_tickets.json")
ARCHIVO_ASIGNACIONES.write_text("[]", encoding="utf-8")

### Tarea 1 — Crear `consultar_colas()`

Debe devolver un diccionario con `ok` y las colas disponibles. No recibe parámetros y no modifica datos.

In [ ]:
def consultar_colas():
    # TODO: usá consultar_menu() como referencia
    pass

### Tarea 2 — Crear `asignar_ticket()` 

La función debe:

1. comprobar que `ticket_id` exista en `TICKETS`;
2. comprobar que `cola` exista en `COLAS`;
3. leer la lista guardada en `ARCHIVO_ASIGNACIONES`;
4. agregar `{"ticket_id": ..., "cola": ...}`;
5. volver a guardar el JSON y devolver un resultado con `ok`.

In [ ]:
def asignar_ticket(ticket_id, cola):
    # TODO: validar ticket_id
    # TODO: validar cola
    # TODO: leer, agregar y guardar la asignación
    pass

### Una buena consulta para la IA

> Necesito completar una función Python breve llamada `asignar_ticket(ticket_id, cola)`. Los tickets y las colas están en dos diccionarios. Las asignaciones se guardan como una lista dentro de un archivo JSON. La función debe validar ambos valores, agregar la asignación y devolver un diccionario con `ok`. No agregues librerías. Explicá cada bloque y proponé tres pruebas.

Revisá la respuesta, eliminá lo innecesario y adaptala a los nombres del notebook.

### Tarea 3 — Registrar y describir las herramientas (10 minutos)

Completá el registro y el `system prompt`. La descripción debe indicar nombre, propósito y parámetros de cada herramienta.

In [ ]:
HERRAMIENTAS_TICKETS = {
    # TODO: registrar consultar_colas
    # TODO: registrar asignar_ticket
}

SYSTEM_PROMPT_TICKETS = """
Sos un agente que organiza tickets de una tienda online.

HERRAMIENTAS DISPONIBLES:
- TODO: describir consultar_colas
- TODO: describir asignar_ticket

Respondé únicamente JSON con este formato:
{"herramienta": "nombre o null", "argumentos": {}}
""".strip()

### Tarea 4 — Probar el agente (5 minutos)

Probá como mínimo:

- una pregunta sobre las colas disponibles;
- una solicitud para asignar `T001` a `pagos`;
- un ticket inexistente;
- una cola inexistente.

In [ ]:
# Ejemplo del recorrido completo
# mensaje = "Asigná el ticket T001 a la cola pagos"
# decision = decidir_con_llm(mensaje, SYSTEM_PROMPT_TICKETS)
# resultado = ejecutar(decision, HERRAMIENTAS_TICKETS)
# print(decision)
# print(resultado)

# TODO: agregá las otras pruebas

## Entrega: explicar las decisiones

Respondé brevemente:

1. ¿Qué hace cada función?
2. ¿Qué parámetros recibe y qué devuelve?
3. ¿Qué parte propuso la IA y qué modificaron ustedes?
4. ¿Cómo describieron cada herramienta en el `system prompt`?
5. ¿Qué validaciones realiza Python antes de escribir el archivo?
6. ¿Qué ocurriría si el LLM inventara una herramienta no registrada?

### Criterio de logro

Las dos funciones son breves, los casos principales funcionan y el equipo puede explicar el código y las decisiones sin depender de la IA que lo generó.

---
## ✅ Cierre

Para dar una herramienta a un agente:

1. construimos una función clara;
2. la registramos en Python;
3. describimos nombre, propósito y parámetros en el `system prompt`;
4. dejamos que el LLM proponga cuándo usarla;
5. permitimos que Python controle la ejecución.

> Describir una herramienta no concede acceso. El acceso existe solamente cuando Python registra y ejecuta la función.